# Fine-tune PaddleOCR cho tiếng Việt với hiệu suất cao

Notebook này được tối ưu để chạy trên **Google Colab** hoặc **Kaggle** với GPU mạnh mẽ (T4, P100, A100).

### Các bước thực hiện:
1. Cài đặt môi trường (PaddlePaddle GPU, PaddleOCR).
2. Clone mã nguồn và chuẩn bị dữ liệu.
3. Chạy training với `batch_size=64` và `num_workers=4`.

## 1. Cài đặt môi trường

In [ ]:
# Kiểm tra GPU
!nvidia-smi

# Cài đặt PaddlePaddle GPU (phù hợp với CUDA 11.8/12.x trên Colab/Kaggle)
!pip install paddlepaddle-gpu==2.6.2

# Cài đặt PaddleOCR và các thư viện hỗ trợ
!pip install paddleocr lmdb langchain_text_splitters

# Clone dự án và PaddleOCR
!git clone --depth 1 https://github.com/Devhub-Solutions/ppocrv5_vi_finetune.git
!git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git

# Cài đặt phụ thuộc của PaddleOCR
%cd PaddleOCR
!pip install -r requirements.txt
%cd ..

## 2. Chuẩn bị dữ liệu
Nếu bạn đã có file `dataset.zip`, hãy upload lên Colab hoặc Kaggle và giải nén.

In [ ]:
# Giải nén dữ liệu (thay đổi đường dẫn nếu cần)
import os
dataset_path = "/content/dataset.zip" # Đường dẫn trên Colab
if not os.path.exists(dataset_path):
    dataset_path = "/kaggle/input/ppocrv5-vi-dataset/dataset.zip" # Đường dẫn trên Kaggle

if os.path.exists(dataset_path):
    !unzip -o {dataset_path} -d ./ppocrv5_vi_finetune/
else:
    print("Vui lòng upload dataset.zip lên thư mục hiện tại.")

## 3. Chạy Training
Sử dụng `batch_size=64` để tận dụng tối đa GPU RAM.

In [ ]:
%cd ppocrv5_vi_finetune

# Tạo file config với batch_size lớn
!python3 2_finetune_recognition.py --train_dir ./dataset_output --output_dir ./finetuned_models/rec --epochs 100 --batch_size 64 --num_workers 4

# Chạy training chính thức
%cd ../PaddleOCR
import os
os.environ['PYTHONPATH'] = os.getcwd()
!python3 tools/train.py -c /content/ppocrv5_vi_finetune/finetuned_models/rec/config/rec_vi_config.yml

## 4. Export Model
Sau khi training xong, export model sang định dạng inference.

In [ ]:
!python3 tools/export_model.py -c /content/ppocrv5_vi_finetune/finetuned_models/rec/config/rec_vi_config.yml \
    -o Global.pretrained_model=/content/ppocrv5_vi_finetune/finetuned_models/rec/rec_model/best_accuracy \
    Global.save_inference_dir=/content/ppocrv5_vi_finetune/finetuned_models/rec/rec_inference